**HVG count study | catnap_core** ᓚᘏᗢ
-----

How accuracy evolves with the number of genes each node selects, and what kind of node needs more of them. f_statistic re-selected at every node, logit node models down the tree with at most 2000 lbfgs iterations, AIFI, the same 5 donor folds as `hvg_selection.ipynb`.

1. Accuracy, hierarchical precision / recall / F1 and training time against the budget $n$, genes selected at each node against the root's top $n$ reused everywhere (the fixed_global arm of `hvg_selection.ipynb`), every atlas marker as the level to beat.
2. The overlap between each node's top-$n$ and the global top-$n$ (the root's leaf-cut selection), for every $n$.
3. Each node's own accuracy against $n$, and whether its depth, arity, size, children's sizes or balance go with needing more genes.

Output: runs to `runs_quantity/`, figures to `figures_quantity/aifi/`, tables to `tables/aifi/`. The 300-HVG folds, the 2000-HVG fold 0 and the atlas-marker folds were first trained by `hvg_selection.ipynb`, the same models, and are copied here with `quantity.import_run`. 5000 and 10000 genes per node run on fold 0 only, 10000 trained with every node in parallel (`quantity.train_parallel`, checked to reproduce the 25-gene fold 0).

In [ ]:
# --- IMPORTS ---
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
from catnap_core.utils import set_verbosity
from hvgsel import datasets, quantity, runs

set_verbosity(0)

In [ ]:
# --- SETUP ---
DATASET  = "aifi"
FOLDS    = range(5)
GRID     = (10, 25, 50, 100, 200, 300, 500, 1000, 2000)   # genes per node, every fold
WIDE     = (5000, 10000)                                   # genes per node, fold 0 only
MAX_ITER = 2000                                            # lbfgs iterations, catnap's default is 1000
N_MARKERS = 140                                            # atlas markers, every one at every node

DS = datasets.DATASETS[DATASET]
FIG, TEX = Path("figures_quantity") / DATASET, Path("tables") / DATASET
LEVELS = [f"L{k}" for k in range(1, len(DS.label_cols) + 1)]
adata = datasets.load_dataset(DATASET)

Accuracy against the budget
---------------------------

Hierarchical metrics compare the set of labels along each cell's true path with the set along its predicted path: **hF** pools them over cells, **macro hF** averages a per-leaf-class F1, so a population of tens of cells weighs like one of a hundred thousand. **exact** is the share of cells whose deepest label is right.

Genes selected at each node are drawn solid, the root's top $n$ reused at every node dashed, in the same colour, and every atlas marker at every node dotted across: the level a selection has to beat.

In [ ]:
# trained by hvg_selection.ipynb: both arms of the 300-HVG folds and the 2000-HVG fold 0, and the
# atlas markers' fixed_global arm, every marker at every node; copied in once, the sources untouched
for fold in FOLDS:
    for arm in (quantity.ARM, quantity.FIXED):
        quantity.import_run(runs.Run("runs_cv", DATASET, f"f_statistic_0300hvg_logit_fold{fold}"), 300, fold, arm=arm)
    markers = runs.Run("runs_cv", DATASET, f"marker_0000hvg_logit_fold{fold}")
    quantity.import_run(markers, N_MARKERS, fold, arm=quantity.FIXED, tag=markers.tag, selector="marker")
for arm in (quantity.ARM, quantity.FIXED):
    quantity.import_run(runs.Run("runs", DATASET, "f_statistic_2000hvg_logit"), 2000, 0, arm=arm)

In [ ]:
for n in GRID + WIDE:
    for fold in (FOLDS if n in GRID else [0]):
        quantity.train_budget(adata, DATASET, DS.label_cols, DS.config_path, n, fold, DS.donor_col,
                              max_iter=MAX_ITER)
        quantity.train_fixed(adata, DATASET, DS.label_cols, DS.config_path, n, fold, DS.donor_col,
                             max_iter=MAX_ITER)

Convergence
-----------

At catnap's default of 1000 iterations, about 45 % of the node models stop at the cap, the large nodes mostly. The GRID runs' reselect arms and the copied runs were trained at 1000: every node that stopped at the cap is trained again at `MAX_ITER`, from the same cells, classes and genes, and its arm predicted again. A node at the cap after `MAX_ITER` stays there. The capped models are kept under `reselect/models_maxiter1000/`, the predictions and metrics they gave as `predictions_maxiter1000.csv.gz` and `metrics_maxiter1000.json`. Runs trained at `MAX_ITER` from the start have nothing to retrain.

In [ ]:
quantity.retrain_capped(adata, quantity.arm_runs(), DS.label_cols, DS.config_path, DS.donor_col,
                        max_iter=MAX_ITER)

In [ ]:
display(quantity.convergence_table(quantity.arm_runs()))

In [ ]:
metrics = quantity.metrics_table(("runs_quantity",))
uniform = metrics[metrics["node_n"].isna()]   # one budget at every node
uniform.to_csv(TEX / "quantity_metrics.csv", index=False)
quantity.budget_figure(uniform, path=FIG / "quantity_accuracy.pdf", cells_only=True)
quantity.time_figure(uniform, path=FIG / "quantity_accuracy_time.pdf", cells_only=True)
quantity.budget_figure(uniform, path=FIG / "quantity_macrof1.pdf", per_level="macro_f1")
quantity.time_figure(uniform, path=FIG / "quantity_macrof1_time.pdf", per_level="macro_f1")
display(uniform.groupby(["selector", "arm", "n_hvg"])[["hF", "macro_hF", "exact", "accuracy_L3", "macro_f1_L3",
                                                       "fit_seconds"]].agg(["mean", "std"]).round(4))

Overlap with the global selection
---------------------------------

For every split node and every $n$: the share of the node's top-$n$ (f_statistic on its children) also in the global top-$n$ (f_statistic at the root on the leaves, the fixed_global selection). Each node's F vector is scored once per fold and cached under `runs_quantity/<dataset>/_scores/`, the same vectors the budget runs select from.

In [ ]:
curves = []
for fold in FOLDS:
    split = runs.make_split(adata, DS.label_cols, DS.config_path, donor_col=DS.donor_col,
                            n_folds=5, fold=fold, seed=0)
    genes, scores = quantity.fold_scores(adata, DS.label_cols, DS.config_path, split)
    ns = np.unique(np.concatenate([np.arange(1, 201), np.round(np.logspace(np.log10(200),
                                                                np.log10(len(genes)), 150)).astype(int)]))
    reference = quantity.global_ranking("runs_cv", DATASET, split.key)
    curves.append(quantity.overlap_curves(quantity.node_rankings(genes, scores), reference, ns)
                  .assign(fold=fold))
curves = pd.concat(curves, ignore_index=True)
features = quantity.overlap_features(curves)
features.to_csv(TEX / "quantity_overlap_features.csv", index=False)

quantity.overlap_figure(curves, features, len(genes), path=FIG / "quantity_overlap_curves.pdf")
display(features.groupby(["depth", "branch", "node"])[["n_first", "n50"]].median())

Accuracy per node
-----------------

Each split node's own decision at every budget: its saved model replayed on every held-out cell truly below it, whether or not its parent routed the cell there, so the nodes above do not choose which cells it is scored on. Accuracy, and macro-F1 over its children, which weighs a rare child like an abundant one. The replays are cached under `runs_quantity/<dataset>/_decisions/`. A node needs the $n$ at which the mean over folds first reaches 90 % of its gain from the smallest budget to its best, against its shape, read off the true labels of all folds' held-out cells (the whole dataset).

In [ ]:
budgets  = quantity.budget_runs()
per_node = quantity.decision_curves(adata, budgets, DS.label_cols)
shape    = quantity.node_properties(quantity.held_out_truth(budgets, LEVELS), LEVELS, DS.config_path)
needs    = {metric: quantity.saturation(per_node, metric) for metric in ("accuracy", "macro_f1")}
per_node.to_csv(TEX / "quantity_node_metrics.csv", index=False)
table = shape.merge(needs["accuracy"].add_prefix("accuracy_").rename(columns={"accuracy_path": "path"}), on="path") \
             .merge(needs["macro_f1"].add_prefix("macro_f1_").rename(columns={"macro_f1_path": "path"}), on="path")
table.to_csv(TEX / "quantity_node_needs.csv", index=False)

quantity.node_curves_figure(per_node, shape, needs, path=FIG / "quantity_node_curves.pdf")
quantity.node_needs_figure(needs, shape, path=FIG / "quantity_node_needs.pdf")
display(table.set_index("node")[["depth", "arity", "size", "smallest_child", "balance",
                                 "accuracy_n90", "accuracy_best", "macro_f1_n90", "macro_f1_best"]].round(3))

Budgets tuned per node
----------------------

Every node chooses its own number of genes on donors its fold's training holds out: the training donors dealt again into 5 groups, one held out for validation, every node's F vectors scored on the others alone, every node trained there at each budget of `quantity.TUNE_GRID` (10 to 1000) and its own decision scored on the validation cells below it, macro-F1 over its children. Each node takes its budget of best validation macro-F1, the smaller on a tie; the tree is then trained on the whole training split with those budgets and scored on the held-out donors, against 1000 and 2000 genes at every node. The validation curves and the budgets chosen are kept under `runs_quantity/<dataset>/_tuning/`.

In [ ]:
quantity.train_tuned(adata, DATASET, DS.label_cols, DS.config_path, DS.donor_col, folds=FOLDS, max_iter=MAX_ITER)

In [ ]:
quantity.tuned_figure(metrics, path=FIG / "quantity_tuned.pdf")
tuned = metrics[metrics["variant"] == quantity.TUNED]
display(tuned[["fold", "hF", "macro_hF", "accuracy_L3", "macro_f1_L1", "macro_f1_L2", "macro_f1_L3", "n_genes_trained"]].round(4))